# 01 Data Inventory

Notebook-first article reproduction. Calculations are kept in notebook cells.

## Inventory
Validate the local input table and write machine-readable provenance tables.

In [1]:

from pathlib import Path
import json
import math
import pickle
import shutil
import subprocess
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"
MANUSCRIPT_FIGURES = FIGURES / "manuscript"
MODELS_DIR = RESULTS / "models"
SPLITS_DIR = RESULTS / "splits"
MANUSCRIPT = ROOT / "manuscript"
GENERATED = MANUSCRIPT / "generated"

for path in [TABLES, MANUSCRIPT_FIGURES, MODELS_DIR, SPLITS_DIR, GENERATED]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_COL = "-lgLD50, mol/kg"
CLUSTER_COL = "Butina_clusters"
BBB_FILTER_COL = "bbb_rule_pass"
LIGAND_ID_COL = "ligand_id"
FP_PREFIX = "FP_"
RANDOM_STATE = 42

PROTEINS = [
    "1m2z", "1pbq", "1xoq", "2rh1", "2vt4", "2ydo", "2z5x", "3b66", "3kk6", "3ln1",
    "3rze", "4djh", "4ey7", "4iar", "4mqs", "4n6h", "5cxv", "5i71", "5tvn", "5u09",
    "5va1", "6cm4", "6kpf", "6kux", "6lqa", "6pdj", "6x3x", "6y1z", "7f8y", "7kwe",
    "7ljd", "7wc9", "7xnk", "7ym8", "8e9y", "8ef6", "8fhs", "8pjk", "8st0", "8wty",
    "8xvk", "8yn3", "9eo4", "V1A",
]
MODEL_ORDER = ["Baseline", "PCA", "ADME", "Plain", "BBB pass"]


In [2]:

df_path = DATA / "df_final.csv"
if not df_path.exists():
    raise FileNotFoundError(df_path)

df = pd.read_csv(df_path)
print("df_final.csv shape:", df.shape)
if tuple(df.shape) != (12651, 2238):
    raise AssertionError(f"Unexpected df_final.csv shape: {df.shape}")

required = {TARGET_COL, CLUSTER_COL, BBB_FILTER_COL, LIGAND_ID_COL, "MW", "MW, g/mol", "logP", *PROTEINS}
missing = sorted(required.difference(df.columns))
if missing:
    raise AssertionError(f"Missing required columns: {missing}")

fp_cols = sorted([c for c in df.columns if c.startswith(FP_PREFIX)], key=lambda x: int(x.split("_")[1]))
if len(fp_cols) != 2048:
    raise AssertionError(f"Expected 2048 fingerprint columns, found {len(fp_cols)}")

protein_nan_total = int(df[PROTEINS].isna().sum().sum())
protein_rows_any_nan = int(df[PROTEINS].isna().any(axis=1).sum())
print("protein NaN cells:", protein_nan_total)
print("rows with any protein NaN:", protein_rows_any_nan)

inventory = pd.DataFrame([
    {"item": "n_rows", "value": int(df.shape[0])},
    {"item": "n_columns", "value": int(df.shape[1])},
    {"item": "n_fingerprint_columns", "value": int(len(fp_cols))},
    {"item": "n_protein_columns", "value": int(len(PROTEINS))},
    {"item": "protein_nan_cells", "value": protein_nan_total},
    {"item": "rows_with_any_protein_nan", "value": protein_rows_any_nan},
    {"item": "target_nonmissing", "value": int(df[TARGET_COL].notna().sum())},
    {"item": "unique_ligand_id", "value": int(df[LIGAND_ID_COL].nunique())},
])
inventory.to_csv(TABLES / "data_inventory.csv", index=False)

protein_nan = pd.DataFrame({
    "protein": PROTEINS,
    "n_nan": [int(df[c].isna().sum()) for c in PROTEINS],
    "n_nonmissing": [int(df[c].notna().sum()) for c in PROTEINS],
})
protein_nan.to_csv(TABLES / "protein_nan_summary.csv", index=False)

feature_inventory = pd.DataFrame([
    {"feature_group": "fingerprint", "n_columns": len(fp_cols), "columns": ";".join(fp_cols)},
    {"feature_group": "protein", "n_columns": len(PROTEINS), "columns": ";".join(PROTEINS)},
    {"feature_group": "plain_physchem", "n_columns": 2, "columns": "MW, g/mol;logP"},
    {"feature_group": "conformal_plain_physchem", "n_columns": 2, "columns": "MW;logP"},
])
feature_inventory.to_csv(TABLES / "feature_inventory.csv", index=False)
print(inventory.to_string(index=False))


df_final.csv shape: (12651, 2238)
protein NaN cells: 8159
rows with any protein NaN: 3354
                     item  value
                   n_rows  12651
                n_columns   2238
    n_fingerprint_columns   2048
        n_protein_columns     44
        protein_nan_cells   8159
rows_with_any_protein_nan   3354
        target_nonmissing  12651
         unique_ligand_id  12651


In [3]:

support_rows = []
support_files = [
    ("data/excluded_molecules.csv", DATA / "excluded_molecules.csv"),
    ("data/receptor_panel_annotation.csv", DATA / "receptor_panel_annotation.csv"),
    ("results/splits/split_indices.csv", SPLITS_DIR / "split_indices.csv"),
]
for name, path in support_files:
    support_rows.append({"file": name, "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0})
    if not path.exists():
        raise FileNotFoundError(path)

pd.DataFrame(support_rows).to_csv(TABLES / "support_files_inventory.csv", index=False)
print(pd.DataFrame(support_rows).to_string(index=False))


                              file  exists   bytes
       data/excluded_molecules.csv    True     254
data/receptor_panel_annotation.csv    True    2881
  results/splits/split_indices.csv    True 5278168
